# Import

In [ ]:
import os
from datetime import datetime
from utils import get_datalake_logger
from scripts.mount import ensure_mount
from scripts.write_df_to_parquet import write_df_to_parquet
from scripts.utils_gold import *

# Config

In [ ]:
STORAGE_ACCOUNT_NAME = os.environ["STORAGE_ACCOUNT_NAME"]
FILESYSTEM_NAME_GOLD = os.environ["CONTAINER_GOLD"]
SECRET_SCOPE_NAME = os.environ["SECRET_SCOPE_NAME"]
SECRET_KEY_NAME = os.environ["SECRET_KEY_NAME"]
MOUNT_POINT_GOLD = "/mnt/donnees-qualite-eau-gold"

# Configuration du logging

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = f"notebooks/04_aggregate_data_to_gold/run_{timestamp}.log"
logger = get_datalake_logger(
    logger_name="04_aggregate_data_to_gold",
    account_name=STORAGE_ACCOUNT_NAME,
    account_key=dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=SECRET_KEY_NAME),
    filesystem_name="logs",
    log_path=log_path
)

# Vérifier / Créer le montage du Data Lake si nécessaire

In [ ]:
# Crée les montages si besoin
ensure_mount(
    mount_point=MOUNT_POINT_GOLD,
    container_name=FILESYSTEM_NAME_GOLD,
    secret_scope_name=SECRET_SCOPE_NAME,
    secret_key_name=SECRET_KEY_NAME,
    storage_account_name=STORAGE_ACCOUNT_NAME,
    logger=logger
)

# Config Schema

In [ ]:
logger.info("Vérification de l’existence du schéma 'gold'...")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
logger.info("Le schéma 'gold' est prêt à être utilisé.")

# Récupération des données du schema Silver

In [ ]:
df_plv_silver = spark.table("silver.dis_plv")
df_result_silver = spark.table("silver.dis_result")
df_com_silver = spark.table("silver.dis_com")

# Creation des table de dimension

## Info reseaux et communes

In [ ]:
try:
    logger.info("Construction du DataFrame dim_commune à partir de df_com_silver")
    
    window_spec = Window.partitionBy("insee_commune").orderBy(F.col("annee").desc())
    
    dim_commune = (
        df_com_silver
        .withColumn("row_number", F.row_number().over(window_spec))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
        .select("insee_commune", "nom_commune", "annee")
    ).drop("annee")
    
    logger.info("DataFrame dim_commune construit avec succès.")
    
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame dim_commune: {e}")

In [ ]:
# Table des réseaux
try:
    logger.info("Construction du DataFrame dim_reseau à partir de df_com_silver")
    window_spec_reseau = Window.partitionBy("cd_reseau").orderBy(F.col("annee").desc())

    dim_reseau = (
        df_com_silver
        .withColumn("row_number", F.row_number().over(window_spec_reseau))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
        .select("cd_reseau", "nom_reseau", "debut_alim", "annee")
    ).drop("annee")
    logger.info("DataFrame dim_reseau construit avec succès.")
    
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame dim_reseau: {e}")

In [ ]:
# Table de liaison : commune ↔ réseau ↔ quartier
try:
    logger.info("Construction du DataFrame dim_affectation_reseau à partir de df_com_silver")
    dim_affectation_reseau = df_com_silver.select(
        "insee_commune", "cd_reseau", "quartier"
    ).distinct()
    logger.info("DataFrame dim_affectation_reseau construit avec succès.")
        
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame dim_affectation_reseau: {e}")

## Parametres

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_com_silver")
    window_param = Window.partitionBy("cd_parametre").orderBy(F.col("annee").desc())
    # Table des infos des paramètres 
    dim_parametre_info = (
        df_result_silver
        .withColumn("row_number", F.row_number().over(window_param))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
        .select(
            "cd_parametre",
            "lib_parametre",
            "cd_parametre_sise_eaux",
            "is_qualitatif",
            "cd_unite_reference",
            "min_val_ref",
            "max_val_ref",
            "valeur_limite",
        )
    )
    logger.info("DataFrame dim_parametre_info construit avec succès.")
        
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame dim_parametre_info: {e}")

In [ ]:
# Table des unités
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de dim_unite")
    dim_unite = (
        df_result_silver
        .withColumn("row_number", F.row_number().over(window_param))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
        .select(
            "cd_unite_reference_sise_eaux",
            "cd_unite_reference"
        )
    )
    logger.info("DataFrame dim_unite construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame dim_unite: {e}")

In [ ]:
# Table des source d'analyse

try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_source_analyse")
    df_dim_source_analyse = (
        df_result_silver
        .select(
            "reference_prel",
            "cd_parametre",
            "cd_ana_labo",
            "cd_cas_param",
            "annee",
            "is_labo"
        )
        .dropDuplicates()  # ou .distinct(), pour garder 1 ligne unique
    )
    logger.info("DataFrame dim_udf_dim_source_analysenite construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_source_analyse: {e}")

## Info prelevement

In [ ]:
# Table des prélèvements par département
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_prel_dept")
    df_dim_prel_dept = (
        df_result_silver
        .select("cd_dept", "reference_prel","annee")
        .dropDuplicates()
    )
    logger.info("DataFrame df_dim_prel_dept construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_prel_dept: {e}")

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_prel_date")
    df_dim_prel_date = (
        df_plv_silver
        .select(
            "reference_prel",
            "date_prel",
            "heure_prel",
            "annee"
        )
        .dropDuplicates(["reference_prel", "annee"])
    )
    logger.info("DataFrame df_dim_prel_date construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_prel_date: {e}")

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_unite_gestion")
    df_dim_unite_gestion = (
        df_plv_silver
        .select("unite_gestion")
        .dropna()
        .dropDuplicates()
        .withColumn("id_unite_gestion", F.monotonically_increasing_id())
    )
    logger.info("DataFrame df_dim_unite_gestion construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_unite_gestion: {e}")

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_organisme_exploitant")
    df_dim_organisme_exploitant = (
        df_plv_silver
        .select("organisme_exploitant")
        .dropna()
        .dropDuplicates()
        .withColumn("id_organisme_exploitant", F.monotonically_increasing_id())
    )

    logger.info("DataFrame df_dim_organisme_exploitant construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_organisme_exploitant: {e}")

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_maitre_ouvrage")
    df_dim_maitre_ouvrage = (
        df_plv_silver
        .select("maitre_ouvrage")
        .dropna()
        .dropDuplicates()
        .withColumn("id_maitre_ouvrage", F.monotonically_increasing_id())
    )

    logger.info("DataFrame df_dim_maitre_ouvrage construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_maitre_ouvrage: {e}")

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_info_exploitant")
    df_dim_info_exploitant = (
        df_plv_silver
        .select(
            "reference_prel",
            "annee",
            "unite_gestion",
            "organisme_exploitant",
            "maitre_ouvrage"
        )
        .dropDuplicates(["reference_prel", "annee"])
    )

    df_dim_info_exploitant = (
        df_dim_info_exploitant
        .join(df_dim_unite_gestion, ["unite_gestion"], "left")
        .join(df_dim_organisme_exploitant, ["organisme_exploitant"], "left")
        .join(df_dim_maitre_ouvrage, ["maitre_ouvrage"], "left")
        .select(
            "reference_prel",
            "annee",
            "id_unite_gestion",
            "id_organisme_exploitant",
            "id_maitre_ouvrage"
        )
    )

    logger.info("DataFrame df_dim_info_exploitant construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_info_exploitant: {e}")

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_dim_info_reseau_plv")
    df_dim_info_reseau_plv = (
        df_plv_silver
        .select(
            "reference_prel",
            "annee",
            "cd_reseau",
            "insee_commune_princ",
            "cd_reseau_amont",
            "pourcent_debit"
        )
        .dropDuplicates(["reference_prel", "annee", "cd_reseau"])
        .withColumnRenamed("insee_commune_princ", "insee_commune")
    )
    logger.info("DataFrame df_dim_info_reseau_plv construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_dim_info_reseau_plv: {e}")

# Creation des tables de fait

In [ ]:
# Table des résultats
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_fact_resultats")
    df_fact_resultats = (
        df_result_silver
        .select(
            "reference_prel",
            "annee",
            "cd_parametre",
            "val_quantitatif",
            "val_qualitatif"
        ).dropDuplicates()
    )
    logger.info("DataFrame df_fact_resultats construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_fact_resultats: {e}")

In [ ]:
try:
    logger.info("Construction du DataFrame dim_parametre_info à partir de df_fact_prel_bilan")
    df_fact_prel_bilan = (
        df_plv_silver
        .select(
            "reference_prel",
            "annee",
            "plv_conformite_bacterio",
            "plv_conformite_chimique",
            "plv_conformite_reference_bact",
            "plv_conformite_reference_chim",
            "conclusion_prel"
        )
        .dropDuplicates(["reference_prel", "annee"])
    )
    logger.info("DataFrame df_fact_prel_bilan construit avec succès.")
except Exception as e:
    logger.error(f"Erreur lors de la construction du DataFrame df_fact_prel_bilan: {e}")

# Insertion en table

In [ ]:
upsert_configs = [
    # dim_commune
    {
        "df": dim_commune,
        "table_name": "gold.dim_commune",
        "partition_cols": [],
        "merge_condition": "t.insee_commune = s.insee_commune",
        "update_condition": "t.nom_commune <> s.nom_commune",
        "update_cols": {"nom_commune": "s.nom_commune"}
    },
    # dim_reseau
    {
        "df": dim_reseau,
        "table_name": "gold.dim_reseau",
        "partition_cols": [],
        "merge_condition": "t.cd_reseau = s.cd_reseau",
        "update_condition": "t.nom_reseau <> s.nom_reseau OR t.debut_alim <> s.debut_alim",
        "update_cols": {"nom_reseau": "s.nom_reseau", "debut_alim": "s.debut_alim"}
    },
    # dim_affectation_reseau
    {
        "df": dim_affectation_reseau,
        "table_name": "gold.dim_affectation_reseau",
        "partition_cols": [],
        "merge_condition": "t.insee_commune = s.insee_commune AND t.cd_reseau = s.cd_reseau AND t.quartier = s.quartier",
        "update_condition": "",
        "update_cols": {}
    },
    # dim_parametre_info
    {
        "df": dim_parametre_info,
        "table_name": "gold.dim_parametre_info",
        "partition_cols": [],
        "merge_condition": "t.cd_parametre = s.cd_parametre",
        "update_condition": "t.lib_parametre <> s.lib_parametre OR t.cd_parametre_sise_eaux <> s.cd_parametre_sise_eaux OR t.is_qualitatif <> s.is_qualitatif OR t.cd_unite_reference <> s.cd_unite_reference OR t.min_val_ref <> s.min_val_ref OR t.max_val_ref <> s.max_val_ref OR t.valeur_limite <> s.valeur_limite",
        "update_cols": {
            "lib_parametre": "s.lib_parametre",
            "cd_parametre_sise_eaux": "s.cd_parametre_sise_eaux",
            "is_qualitatif": "s.is_qualitatif",
            "cd_unite_reference": "s.cd_unite_reference",
            "min_val_ref": "s.min_val_ref",
            "max_val_ref": "s.max_val_ref",
            "valeur_limite": "s.valeur_limite"
        }
    },
    # dim_unite
    {
        "df": dim_unite,
        "table_name": "gold.dim_unite",
        "partition_cols": [],
        "merge_condition": "t.cd_unite_reference_sise_eaux = s.cd_unite_reference_sise_eaux",
        "update_condition": "t.cd_unite_reference <> s.cd_unite_reference",
        "update_cols": {"cd_unite_reference": "s.cd_unite_reference"}
    },
    # dim_prel_date
    {
        "df": df_dim_prel_date,
        "table_name": "gold.dim_prel_date",
        "partition_cols": ["annee"],
        "merge_condition": "t.reference_prel = s.reference_prel AND t.annee = s.annee",
        "update_condition": "t.date_prel <> s.date_prel OR t.heure_prel <> s.heure_prel",
        "update_cols": {
            "date_prel": "s.date_prel",
            "heure_prel": "s.heure_prel"
        }
    },
    # dim_prel_dept
    {
        "df": df_dim_prel_dept,
        "table_name": "gold.dim_prel_dept",
        "partition_cols": ["annee"],
        "merge_condition": "t.reference_prel = s.reference_prel AND t.annee = s.annee",
        "update_condition": "t.cd_dept <> s.cd_dept",
        "update_cols": {"cd_dept": "s.cd_dept"}
    },
    # dim_source_analyse
    {
        "df": df_dim_source_analyse,
        "table_name": "gold.dim_source_analyse",
        "partition_cols": ["annee"],
        "merge_condition": "t.reference_prel = s.reference_prel AND t.annee = s.annee AND t.cd_parametre = s.cd_parametre",
        "update_condition": "t.cd_ana_labo <> s.cd_ana_labo OR t.cd_cas_param <> s.cd_cas_param OR t.is_labo <> s.is_labo",
        "update_cols": {
            "cd_ana_labo": "s.cd_ana_labo",
            "cd_cas_param": "s.cd_cas_param",
            "is_labo": "s.is_labo"
        }
    },
    # dim_unite_gestion
    {
        "df": df_dim_unite_gestion,
        "table_name": "gold.dim_unite_gestion",
        "partition_cols": [],
        "merge_condition": "t.unite_gestion = s.unite_gestion",
        "update_condition": "",
        "update_cols": {}
    },
    # dim_organisme_exploitant
    {
        "df": df_dim_organisme_exploitant,
        "table_name": "gold.dim_organisme_exploitant",
        "partition_cols": [],
        "merge_condition": "t.organisme_exploitant = s.organisme_exploitant",
        "update_condition": "",
        "update_cols": {}
    },
    # dim_maitre_ouvrage
    {
        "df": df_dim_maitre_ouvrage,
        "table_name": "gold.dim_maitre_ouvrage",
        "partition_cols": [],
        "merge_condition": "t.maitre_ouvrage = s.maitre_ouvrage",
        "update_condition": "",
        "update_cols": {}
    },
    # dim_info_exploitant
    {
        "df": df_dim_info_exploitant,
        "table_name": "gold.dim_info_exploitant",
        "partition_cols": ["annee"],
        "merge_condition": "t.reference_prel = s.reference_prel AND t.annee = s.annee",
        "update_condition": "t.id_unite_gestion <> s.id_unite_gestion OR t.id_organisme_exploitant <> s.id_organisme_exploitant OR t.id_maitre_ouvrage <> s.id_maitre_ouvrage",
        "update_cols": {
            "id_unite_gestion": "s.id_unite_gestion",
            "id_organisme_exploitant": "s.id_organisme_exploitant",
            "id_maitre_ouvrage": "s.id_maitre_ouvrage"
        }
    },
    # dim_info_reseau_plv
    {
        "df": df_dim_info_reseau_plv,
        "table_name": "gold.dim_info_reseau_plv",
        "partition_cols": ["annee"],
        "merge_condition": "t.reference_prel = s.reference_prel AND t.annee = s.annee AND t.cd_reseau = s.cd_reseau",
        "update_condition": "t.insee_commune <> s.insee_commune OR t.cd_reseau_amont <> s.cd_reseau_amont OR t.pourcent_debit <> s.pourcent_debit",
        "update_cols": {
            "insee_commune": "s.insee_commune",
            "cd_reseau_amont": "s.cd_reseau_amont",
            "pourcent_debit": "s.pourcent_debit"
        }
    },
    # df_fact_resultats
    {
        "df": df_fact_resultats,
        "table_name": "gold.fact_resultats",
        "partition_cols": ["annee"],
        "merge_condition": "t.reference_prel = s.reference_prel AND t.annee = s.annee AND t.cd_parametre = s.cd_parametre",
        "update_condition": "t.val_quantitatif <> s.val_quantitatif OR t.val_qualitatif <> s.val_qualitatif",
        "update_cols": {
            "val_quantitatif": "s.val_quantitatif",
            "val_qualitatif": "s.val_qualitatif"
        }
    },
    # df_fact_prel_bilan
    {
        "df": df_fact_prel_bilan,
        "table_name": "gold.fact_prel_bilan",
        "partition_cols": ["annee"],
        "merge_condition": "t.reference_prel = s.reference_prel AND t.annee = s.annee",
        "update_condition": "t.plv_conformite_bacterio <> s.plv_conformite_bacterio OR t.plv_conformite_chimique <> s.plv_conformite_chimique OR t.plv_conformite_reference_bact <> s.plv_conformite_reference_bact OR t.plv_conformite_reference_chim <> s.plv_conformite_reference_chim OR t.conclusion_prel <> s.conclusion_prel",
        "update_cols": {
            "plv_conformite_bacterio": "s.plv_conformite_bacterio",
            "plv_conformite_chimique": "s.plv_conformite_chimique",
            "plv_conformite_reference_bact": "s.plv_conformite_reference_bact",
            "plv_conformite_reference_chim": "s.plv_conformite_reference_chim",
            "conclusion_prel": "s.conclusion_prel"
        }
    }
]

In [ ]:
# Boucle pour upsert toutes les tables
for config in upsert_configs:
    try:
        upsert_table(
            df_silver=config["df"],
            table_name=config["table_name"],
            partition_cols=config["partition_cols"],
            merge_condition=config["merge_condition"],
            update_condition=config["update_condition"],
            update_cols=config["update_cols"],
            logger=logger
        )
    except Exception as e:
        logger.error(f"Erreur lors de l'upsert de {config['table_name']}: {e}")